# GitHub Copilot / GitHub Models - Ejemplos de uso

Este notebook muestra cómo usar la API de modelos de GitHub para acceder a modelos de IA con tu token de GitHub.

GitHub Models permite usar modelos de OpenAI, Meta, Mistral y otros de forma gratuita (con límites) o con suscripción.

**Modelos disponibles:**
- `gpt-4o` - GPT-4o de OpenAI
- `gpt-4o-mini` - GPT-4o Mini de OpenAI
- `o1-mini` - Modelo razonador de OpenAI
- `meta-llama-3.1-70b-instruct` - LLaMA de Meta
- `mistral-large-2407` - Mistral Large

**Documentación:** https://docs.github.com/en/github-models  
**Obtén tu token en:** https://github.com/settings/tokens

## 1. Instalación de dependencias

In [ ]:
%pip install openai python-dotenv

## 2. Configuración del token de GitHub

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Cargar variables de entorno desde .env
load_dotenv(Path('..') / '.env')

# Endpoint de GitHub Models (compatible con OpenAI)
GITHUB_MODELS_BASE_URL = 'https://models.inference.ai.azure.com'

github_token = os.getenv('GITHUB_TOKEN')
if github_token:
    client = OpenAI(
        api_key=github_token,
        base_url=GITHUB_MODELS_BASE_URL
    )
    print(f'✅ Token de GitHub configurado: {github_token[:8]}...')
else:
    print('❌ GITHUB_TOKEN no encontrado. Configura tu archivo .env')

## 3. Ejemplo básico: Chat con GPT-4o

In [ ]:
respuesta = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'Eres un asistente de programación experto.'},
        {'role': 'user', 'content': '¿Cómo implemento un API REST con FastAPI en Python? Dame un ejemplo básico.'}
    ]
)

print(respuesta.choices[0].message.content)

## 4. Ejemplo: Revisión de código (simulando GitHub Copilot)

In [ ]:
codigo_a_revisar = '''
def suma_lista(lista):
    total = 0
    for i in range(len(lista)):
        total = total + lista[i]
    return total

def buscar_elemento(lista, elemento):
    encontrado = False
    for i in range(len(lista)):
        if lista[i] == elemento:
            encontrado = True
    return encontrado
'''

respuesta = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {
            'role': 'system',
            'content': 'Eres GitHub Copilot, un experto revisor de código Python. '
                       'Identifica mejoras, errores, malas prácticas y sugiere refactorizaciones '
                       'usando Pythonic code y mejores prácticas modernas.'
        },
        {
            'role': 'user',
            'content': f'Por favor revisa este código Python:\n\n```python\n{codigo_a_revisar}\n```'
        }
    ]
)

print(respuesta.choices[0].message.content)

## 5. Ejemplo: Generación de tests unitarios

In [ ]:
funcion = '''
def calcular_descuento(precio: float, porcentaje: float) -> float:
    """Calcula el precio con descuento aplicado."""
    if precio < 0:
        raise ValueError("El precio no puede ser negativo")
    if not 0 <= porcentaje <= 100:
        raise ValueError("El porcentaje debe estar entre 0 y 100")
    return precio * (1 - porcentaje / 100)
'''

respuesta = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {
            'role': 'system',
            'content': 'Eres un experto en testing de Python. Genera tests unitarios '
                       'completos usando pytest, cubriendo casos normales, edge cases y excepciones.'
        },
        {
            'role': 'user',
            'content': f'Genera tests unitarios para esta función:\n\n```python\n{funcion}\n```'
        }
    ]
)

print(respuesta.choices[0].message.content)

## 6. Ejemplo: Comparación de modelos

In [ ]:
pregunta = '¿Cuál es la complejidad temporal de QuickSort y en qué casos es mejor que MergeSort?'

modelos = ['gpt-4o-mini', 'gpt-4o']

for modelo in modelos:
    print(f'\n=== {modelo} ===')
    try:
        respuesta = client.chat.completions.create(
            model=modelo,
            messages=[{'role': 'user', 'content': pregunta}],
            max_tokens=300
        )
        print(respuesta.choices[0].message.content)
    except Exception as e:
        print(f'Error: {e}')